![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🏨 Hospitalization Numbers & ALC Volumes — ETL Pipeline
![Python](https://img.shields.io/badge/Python_3.13-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-CIHI-1B3A6B?style=flat-square)

> **Analytical Question:** How do Nova Scotia's hospitalization volumes and Alternate Level of Care (ALC) rates compare to the national average? How have ALC burdens trended across provinces from 2017 to 2025?

This notebook extracts and cleans **CIHI Table 7 — ALC Volumes and Days** from 8 annual Excel workbooks (2017–18 through 2024–25) and stacks them into a single time-series dataset.

---

## 📦 Source Data

| Files | Content | Fiscal Years |
|-------|---------|-------------|
| `Hospitalization rates & others YYYY.xlsx` × 8 | Table 7: ALC volumes, LOS, and patient days by province | 2017–2018 to 2024–2025 |

**Source:** Canadian Institute for Health Information (CIHI) — Hospital Morbidity Database (HMDB)

---

## 🔧 ETL Pipeline Architecture

```
os.listdir(RAW_DIR)
    └── for each .xlsx file:
          extract fiscal year from filename
          find_alc_sheet()      ← name-based sheet detection ('alc' or 'table 7')
          pd.read_excel()       ← skip 4 header rows, keep first 7 columns
          assign column names   ← CIHI invariant layout
          clean rows            ← drop nulls and footnote rows
    └── pd.concat() → export CSV
```

---

## ⚠️ Known Data Challenges

| Challenge | Detail | Solution |
|-----------|--------|----------|
| Sheet name changes | 2017–2022: `'7. ALC volumes and days'`; 2023+: `'Table 7'` | Match sheets containing `'alc'` or `'table 7'` (case-insensitive) |
| Title rows above data | 4 rows of CIHI title and notes precede the column header | `skiprows=4` on read |
| Footnote rows in data | Rows beginning with `'Note'` appended below data | Filter with `str.contains('Note')` |
| Fiscal year not in data | Each workbook = one fiscal year | Regex extract from filename, insert as column |
| Temp file interference | Excel lock files (`~$filename.xlsx`) in directory | Skip files starting with `~$` |

---


## Step 1 — Imports & Config


In [ ]:
import pandas as pd
import os
import re

RAW_DIR     = r"C:\Users\USER\Downloads\Nova-Scotia_Health_Demographic_Trends\Data\Hospitalization Causes"
OUTPUT_FILE = r"C:\Users\USER\Downloads\Nova-Scotia_Health_Demographic_Trends\Clean\hospitalization_numbers.csv"


## Step 2 — Sheet Detection

CIHI changed its sheet naming convention between fiscal years:
- **2017–2022:** `'7. ALC volumes and days'`
- **2023–2025:** `'Table 7'`

Rather than hardcoding sheet names, we scan each workbook's sheet list for names containing `'alc'` or `'table 7'` (case-insensitive). The first match is used. This is robust to minor naming variations across annual releases.


## Step 3 — Process All Workbooks

For each annual workbook:
1. Skip temp files (`~$`) and non-Excel files
2. Extract the fiscal year end from the filename (`'2018-2019'` → `2019`)
3. Detect the correct ALC sheet by name
4. Read with `skiprows=4` to bypass CIHI title rows
5. Keep the first 7 columns (the Table 7 invariant layout) and assign standard names
6. Drop rows with null `Province/territory` or footnote text
7. Tag with `Fiscal Year`


In [ ]:
all_years = []

for file in os.listdir(RAW_DIR):

    # ── Skip temp and non-Excel files ─────────────────────────────────────
    if file.startswith("~$") or not file.lower().endswith((".xlsx", ".xls")):
        continue

    file_path = os.path.join(RAW_DIR, file)

    # ── Extract fiscal year (ending year) from filename ───────────────────
    match = re.search(r"(\d{4})-(\d{4})", file)
    if not match:
        print(f"⚠️  Skipped (no year): {file}")
        continue

    fiscal_year = int(match.group(2))

    try:
        engine = "xlrd" if file.lower().endswith(".xls") else "openpyxl"
        xls    = pd.ExcelFile(file_path, engine=engine)

        # ── Detect ALC / Table 7 sheet ────────────────────────────────────
        alc_sheets = [
            s for s in xls.sheet_names
            if ("alc" in s.lower()) or ("table 7" in s.lower())
        ]

        if not alc_sheets:
            raise ValueError("ALC / Table 7 sheet not found")

        alc_sheet = alc_sheets[0]

        # ── Read table (skip 4 title rows) ───────────────────────────────
        df = pd.read_excel(xls, sheet_name=alc_sheet, skiprows=4)

        # ── Keep first 7 columns (CIHI invariant layout) ──────────────────
        df = df.iloc[:, :7]
        df.columns = [
            "Province/territory",
            "Number of hospitalizations",
            "Number of hospitalizations with reported ALC days",
            "Proportion of hospitalizations with reported ALC days (%)",
            "Length of stay (in days) Total LOS",
            "Length of stay (in days) ALC LOS",
            "Patient days in ALC (%)",
        ]

        # ── Tag with fiscal year ──────────────────────────────────────────
        df.insert(0, "Fiscal Year", fiscal_year)

        # ── Remove null and footnote rows ─────────────────────────────────
        df = df[df["Province/territory"].notna()]
        df = df[~df["Province/territory"].astype(str).str.contains("Note", case=False, na=False)]

        all_years.append(df)

        print(f"   ✅ Processed: {file} (sheet: {alc_sheet})")

    except Exception as e:
        print(f"   ⚠️  Skipped {file}: {e}")


## Step 4 — Concatenate & Export


In [ ]:
if not all_years:
    raise ValueError("No hospitalization files were successfully processed")

final_df = pd.concat(all_years, ignore_index=True)

final_df.to_csv(OUTPUT_FILE, index=False)

print("\n✅ ETL Complete")
print(f"📁 Output : {OUTPUT_FILE}")
print(f"📊 Rows   : {len(final_df)}")


## ✅ Output Summary

| Output | Location | Rows | Fiscal Years |
|--------|----------|------|-------------|
| Clean CSV | `./Clean/hospitalization_numbers.csv` | ~170 | 2017–18 to 2024–25 |

**Final schema:** `Fiscal Year` \| `Province/territory` \| `Number of hospitalizations` \| `Number of hospitalizations with reported ALC days` \| `Proportion of hospitalizations with reported ALC days (%)` \| `Length of stay (in days) Total LOS` \| `Length of stay (in days) ALC LOS` \| `Patient days in ALC (%)`

**Feeds into:**
- 📊 Power BI — ALC proportion trend: NS vs. Canada (2017–2025)
- 📊 Power BI — Total LOS vs. ALC LOS comparison by province
- 📊 Power BI — Hospitalization volume bar chart with ALC overlay

> ⚠️ **COVID note:** 2020–2021 data reflects pandemic impact — hospitalization volumes dropped significantly due to deferred care and capacity reallocation. Flag this year in all trend visuals.

---

*Nova Scotia Health & Population Analytics · CIHI Hospital Morbidity Database*
